# Data Preprocessing — Solutions

<a target="_blank" href="https://colab.research.google.com/github/LuWidme/uk259/blob/rework/solutions/04_Data_Preprocessing_solutions.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> ⏱️ ~90 min · Day 2

**⚠️ Important:** Try the exercises in `demos/04_Data_Preprocessing.ipynb` yourself first. The struggle is where the learning happens.

These are *reference* solutions with short explanations. Your own working solution may look different and still be correct.

In [ ]:
# === COURSE SETUP — run this cell first! ===
%pip install -q numpy pandas matplotlib seaborn scikit-learn

import os, urllib.request
DATA_URL = "https://raw.githubusercontent.com/LuWidme/uk259/rework/datasets/"
for folder in ("datasets", os.path.join("..", "datasets")):
    os.makedirs(folder, exist_ok=True)
    for fname in ['titanic.csv', 'melb_data.csv', 'Company_data.csv']:
        path = os.path.join(folder, fname)
        if not os.path.exists(path):
            urllib.request.urlretrieve(DATA_URL + fname, path)
print("Setup complete — you are ready to go!")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
df = pd.read_csv('../datasets/melb_data.csv')
print('Loaded', df.shape[0], 'houses')

## Part A — Core Exercises

### Core Exercise 1: Missing Value Analysis

In [ ]:
missing_percentages = df.isnull().mean() * 100
columns_with_high_missing = missing_percentages[missing_percentages > 5]

print("Columns with >5% missing values:")
print(columns_with_high_missing)

### Core Exercise 2: Smart Missing Value Imputation

In [ ]:
df_ex2 = df.copy()
missing_count = df_ex2['Car'].isnull().sum()
median_value = df_ex2['Car'].median()
df_ex2['Car'] = df_ex2['Car'].fillna(median_value)

print(f"Missing values before: {missing_count}")
print(f"Median value used: {median_value}")
print(f"Missing values after: {df_ex2['Car'].isnull().sum()}")

### Core Exercise 3: Standardize Multiple Columns

In [ ]:
df_ex4 = df.copy()
df_ex4['BuildingArea'] = df_ex4['BuildingArea'].fillna(df_ex4['BuildingArea'].median())
scaler = StandardScaler()
df_ex4[['Distance_scaled', 'Landsize_scaled', 'BuildingArea_scaled']] = scaler.fit_transform(
    df_ex4[['Distance', 'Landsize', 'BuildingArea']])

print("Mean and Std of scaled columns:")
for col in ['Distance_scaled', 'Landsize_scaled', 'BuildingArea_scaled']:
    print(f"{col}: mean={df_ex4[col].mean():.10f}, std={df_ex4[col].std():.4f}")

### Core Exercise 4: Normalize Price to [0, 1]

In [ ]:
df_ex5 = df.copy()
normalizer = MinMaxScaler()
df_ex5['Price_normalized'] = normalizer.fit_transform(df_ex5[['Price']])
median_house = df_ex5.iloc[(df_ex5['Price_normalized'] - 0.5).abs().argmin()]

print(f"Normalized price range: [{df_ex5['Price_normalized'].min()}, {df_ex5['Price_normalized'].max()}]")
print(f"\nHouse with price closest to median (0.5):")
print(f"Address: {median_house['Address']}")
print(f"Actual Price: ${median_house['Price']:,.0f}")
print(f"Normalized Price: {median_house['Price_normalized']:.3f}")

### Core Exercise 5: Label Encode Multiple Categorical Columns

In [ ]:
df_ex6 = df.copy()
df_ex6['Method_encoded'] = LabelEncoder().fit_transform(df_ex6['Method'])
df_ex6['SellerG_encoded'] = LabelEncoder().fit_transform(df_ex6['SellerG'])

print("Method encoding mapping:")
print(df_ex6[['Method', 'Method_encoded']].drop_duplicates().sort_values('Method_encoded'))
print(f"\nNumber of unique sellers: {df_ex6['SellerG'].nunique()}")

### Core Exercise 6: One-Hot Encode Property Type

In [ ]:
df_ex7 = df.copy()
original_cols = df_ex7.shape[1]
df_ex7 = pd.get_dummies(df_ex7, columns=['Type'], prefix='PropertyType')
new_cols = df_ex7.shape[1]

print(f"Columns before: {original_cols}")
print(f"Columns after: {new_cols}")
print(f"Columns added: {new_cols - original_cols}")
type_cols = [c for c in df_ex7.columns if c.startswith('PropertyType_')]
print(f"\nNew columns: {type_cols}")
print(df_ex7[type_cols].head(10))

## Part B — Bonus Exercises (optional)

### Bonus Exercise 1: Detect Outliers in Landsize

In [ ]:
Q1 = df['Landsize'].quantile(0.25)
Q3 = df['Landsize'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers = df[(df['Landsize'] < lower_bound) | (df['Landsize'] > upper_bound)]
max_landsize = df['Landsize'].max()

print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
print(f"Outlier bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
print(f"Number of landsize outliers: {len(outliers)}")
print(f"Maximum landsize: {max_landsize} sq.m")

### Bonus Exercise 2: Feature Engineering - Price per Square Meter

In [ ]:
df_ex8 = df.copy()
df_ex8['BuildingArea'] = df_ex8['BuildingArea'].fillna(df_ex8['BuildingArea'].median())
df_ex8 = df_ex8[df_ex8['BuildingArea'] > 0]
df_ex8['Price_per_SqM'] = df_ex8['Price'] / df_ex8['BuildingArea']
most_expensive_per_sqm = df_ex8.loc[df_ex8['Price_per_SqM'].idxmax()]
avg_by_type = df_ex8.groupby('Type')['Price_per_SqM'].mean()

print("Property with highest price per square meter:")
print(f"Address: {most_expensive_per_sqm['Address']}")
print(f"Price/SqM: ${most_expensive_per_sqm['Price_per_SqM']:,.2f}")
print(f"\nAverage price per square meter by property type:")
print(avg_by_type)

### Bonus Exercise 3: Binning - Categorize Distance

In [ ]:
df_ex9 = df.copy()
distance_bins = [0, 5, 10, 15, df_ex9['Distance'].max()]
distance_labels = ['Very Close', 'Close', 'Medium', 'Far']
df_ex9['Distance_Category'] = pd.cut(df_ex9['Distance'], bins=distance_bins,
                                    labels=distance_labels, include_lowest=True)
category_counts = df_ex9['Distance_Category'].value_counts()
avg_price_by_distance = df_ex9.groupby('Distance_Category', observed=True)['Price'].mean()

print("Properties by distance category:")
print(category_counts)
print("\nAverage price by distance category:")
print(avg_price_by_distance)

### Bonus Exercise 4: Handling Outliers

In [ ]:
df_ex11 = df.copy()
Q1 = df_ex11['Price'].quantile(0.25)
Q3 = df_ex11['Price'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
df_no_outliers = df_ex11[(df_ex11['Price'] >= lower) & (df_ex11['Price'] <= upper)]

print("Before removing outliers:")
print(f"  Count: {len(df_ex11)}")
print(f"  Mean: ${df_ex11['Price'].mean():,.0f}")
print(f"  Median: ${df_ex11['Price'].median():,.0f}")
print("\nAfter removing outliers:")
print(f"  Count: {len(df_no_outliers)}")
print(f"  Mean: ${df_no_outliers['Price'].mean():,.0f}")

### Bonus Exercise 5: Complete Preprocessing Pipeline

A compact end-to-end pipeline: select features, impute, encode, scale.

In [ ]:
from sklearn.preprocessing import StandardScaler
pipe = df.copy()
num_cols = ['Rooms', 'Distance', 'Bathroom', 'Landsize']
cat_cols = ['Type']
# 1) keep needed columns + drop rows with missing target
pipe = pipe[num_cols + cat_cols + ['Price']].dropna(subset=['Price'])
# 2) impute numeric with median
for c in num_cols:
    pipe[c] = pipe[c].fillna(pipe[c].median())
# 3) one-hot encode categoricals
pipe = pd.get_dummies(pipe, columns=cat_cols, drop_first=True)
# 4) scale numeric features
pipe[num_cols] = StandardScaler().fit_transform(pipe[num_cols])

print('Final shape:', pipe.shape)
print('Any missing left?', pipe.isnull().sum().sum())
pipe.head()